In [ ]:
# ── 第一格：环境准备（每个 notebook 开头都一样，直接运行）──
from signal_mysql import database as mysql
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# ── 第一格：环境准备（每个 notebook 开头都一样，直接运行）──
import os
# 1. 显式设置 MySQL 连接环境变量（解决 using password: NO 的问题）
os.environ["SIGNAL_MYSQL_USER"] = "signal_app"
os.environ["SIGNAL_MYSQL_PASSWORD"] = "signal123456"
os.environ["SIGNAL_MYSQL_HOST"] = "127.0.0.1"
os.environ["SIGNAL_MYSQL_PORT"] = "3306"
os.environ["SIGNAL_INSIGHT_SCHEMA"] = "signal_insight"

ROOT = Path.cwd().parent                   # notebook 放在 notebooks/ 里，项目根目录是上一级
DB_SCHEMA = __import__("os").environ.get("SIGNAL_INSIGHT_SCHEMA", "signal_insight")
IMG_DIR = ROOT / "image"                   # 出图保存位置
IMG_DIR.mkdir(exist_ok=True)
assert mysql.schema_exists(DB_SCHEMA), f"找不到数据库 {DB_SCHEMA}：把讲师发的 signal.db 放进 data/，并确认是在 notebooks 文件夹里打开的 notebook"

DAY = "2018-10-03"         # 分析日期（D2 做出的决定）
NB = "D01_项目启动与数据接入"
SOURCE = f"数据来源：沈阳移动信令样本（脱敏，2018 年国庆）；脚本：{NB}"

# 出图全局设置
# 注意要写进 font.family（一串具体字体），matplotlib 才会逐字回退；只写 font.sans-serif 时只会用第一个找到的字体
#
# 课程统一使用开源字体，代码里不出现微软雅黑 / 苹方 / 黑体 / 宋体 / Times New Roman 这些商业字体的名字。
# 为什么：字体是受版权保护的软件，随系统授予的是“本机使用”，课件、报告、PPT 发出去就超出了授权范围；
# 学生把作品放进简历或作品集更是对外公开的商业用途。详见 docs/图表规范.md 第 0 节。
FONTS = ['TeX Gyre Termes', # Times New Roman字体平替
         'Noto Serif CJK SC', # 宋体的平替字体：开源的思源宋体可以商用
         'Source Han Sans SC'] # 黑体的平替的字体：开源的思源黑体可以商用
# 思源字体由 Adobe 和 Google 分别发布，名字不同但字形完全一样，不同机器装的发行版不一样，
# 所以把别名接在后面，换台电脑才不会变方框。只保留本机真的装了的——
# 列了没装的字体，matplotlib 每画一张图都会刷一屏 findfont 警告。
from matplotlib import font_manager as fm
_ALIAS = ['Source Han Serif SC', 'Noto Serif SC', 'Noto Sans CJK SC', 'Noto Sans SC',
         'Noto Sans CJK JP', 'Noto Serif CJK JP']
_installed = {f.name for f in fm.fontManager.ttflist}
FONTS = [f for f in FONTS + _ALIAS if f in _installed] + ['DejaVu Sans']  # DejaVu Sans 兜底负号、上下标
if not [f for f in FONTS if f not in ('TeX Gyre Termes', 'DejaVu Sans')]:
    print("⚠️ 本机没有课程用的开源中文字体，图里的中文会变成方框。"
          "安装方法见 docs/图表规范.md 第 0 节。")
plt.rcParams.update({
    "font.family": FONTS, "axes.unicode_minus": True,
    "axes.spines.top": False, "axes.spines.right": False, "axes.spines.left": False,
    "axes.grid": True, "axes.grid.axis": "y", "axes.axisbelow": True, "grid.color": "#E5E7EB",
    "axes.edgecolor": "#9CA3AF", "xtick.color": "#4B5563", "ytick.color": "#4B5563",
    "xtick.major.size": 0, "ytick.major.size": 0, "legend.frameon": False,
    "figure.dpi": 100, "savefig.dpi": 200,
})
# 配色：灰色打底，一种颜色强调一件事，红色只表示警示
MAIN, MAIN_LIGHT, GRAY, RED = "#1D6FA5", "#DCEAF5", "#C4CAD3", "#D64545"
INK, MUTED, FAINT = "#1F2937", "#6B7280", "#9CA3AF"
MAIN_DARK, MAIN_MID, GRAY_DARK, AMBER = "#174E73", "#7FB3D9", "#8E99A6", "#B7791F"
AREA_COLOR = {"西塔": MAIN, "太原街商圈": "#D9822B", "中街商圈": "#3B8C66",  # 同一区域在所有图里同色
              "沈阳北站": "#7A5EA8", "沈阳世博园": "#B04A6A", "棋盘山风景区": "#6B7F2A"}
SEG_COLOR = {"市内来访": MAIN, "周边居民": MAIN_MID, "外地来客": "#D9822B", "市外来访": "#D9822B",
             "过夜地不明": GRAY}  # “不明”永远是灰色：不是一类人，是不知道

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
con = mysql.connect(DB_SCHEMA)           # 整个 notebook 共用一个数据库连接

In [ ]:
# 自检：版本、中文字体、数据库大小
import sys, matplotlib, seaborn
from matplotlib import font_manager

installed = {f.name for f in font_manager.fontManager.ttflist}
cn_font = next((f for f in plt.rcParams["font.family"] if f in installed and f != "DejaVu Sans"), None)

# 查询 MySQL 中 signal_insight 数据库的数据量大小（MB）
db_size_sql = f"""
SELECT ROUND(SUM(data_length + index_length) / 1024 / 1024, 0)
FROM information_schema.tables
WHERE table_schema = '{DB_SCHEMA}';
"""
db_size = con.execute(db_size_sql).fetchone()[0] or 0

print(f"Python      {sys.version.split()[0]}")
print(f"pandas      {pd.__version__}")
print(f"numpy       {np.__version__}")
print(f"matplotlib {matplotlib.__version__}")
print(f"seaborn {seaborn.__version__}")
print(f"MySQL       8.0.16+")
print(f"中文字体 {cn_font}")
print(f"数据库         {DB_SCHEMA}（{int(db_size)} MB）")

In [ ]:
# 对账：环境检查
assert cn_font is not None, "没有找到中文字体，图里的中文会变成方框：见助教报错对照表"
print("✅ 环境检查通过")

In [ ]:
# ── 代码格 4 · 需求卡 1.3：连上数据库，先看库里有什么 ──
from signal_mysql import database as mysql
# 这一格在做什么：列出库里有哪些表、各有多少行——拿到一个陌生的库，第一件事就是这个。
# sqlite_master 是 SQLite 自带的系统表，记着库里所有对象；换成 MySQL 就是 information_schema.tables，
# 换成 Hive 就是 SHOW TABLES。工具不同，“先看库里有什么”这个动作是一样的。
tables = mysql.read_frame('SELECT table_name AS name FROM information_schema.tables WHERE table_schema=DATABASE() ORDER BY table_name', con)
# 行数要一张表一张表地查（COUNT(*) 没法一次查完所有表），所以这里用列表推导逐张查
tables["行数"] = [mysql.read_frame(f"SELECT COUNT(*) AS n FROM {t}", con)["n"][0] for t in tables["name"]]
tables

In [ ]:
# ── 代码格 5 · 需求卡 1.3：把整张原始信令表读进 pandas ──
from signal_mysql import database as mysql
# 这一格在做什么：把整张原始信令表读进内存。
# 53 万行 × 10 列，占内存约 200 MB，笔记本扛得住，所以可以一次全读。
# 讲台提示：这里要点一句“什么时候不能这么干”——上亿行的表必须在 SQL 里先筛选、先聚合，
# 再把结果读进 pandas。D12 讲慢查询和索引时会回到这个话题。
raw = mysql.read_frame("SELECT * FROM ods_signal", con)
print(raw.shape)   # (行数, 列数)
raw.head()         # 先看前 5 行，确认列名和内容对得上预期

In [ ]:
# ── 代码格 6 · 需求卡 1.3 对账：行数和列数 ──
assert raw.shape == (532_828, 10), f"行列数不对：{raw.shape}，应该是 (532828, 10)"
print("✅ 对账通过：532,828 行 × 10 列")

In [ ]:
# ── 代码格 7 · 需求卡 1.4：数据画像，每个字段长什么样 ──
def profile(df):
    """数据画像：每个字段一行——类型、缺失数（空值 + 空白字符串）、缺失率、唯一值个数、示例值。"""
    rows = []
    for col in df.columns:
        s = df[col]
        n_null = int(s.isna().sum())
        is_text = pd.api.types.is_string_dtype(s) or s.dtype == object
        n_blank = int(s.astype("string").str.strip().eq("").sum()) if is_text else 0
        rows.append({
            "field": col,
            "dtype": str(s.dtype),
            "n_missing": n_null + n_blank,
            "missing_rate": (n_null + n_blank) / len(s),
            "n_unique": int(s.nunique()),
            "example": s.dropna().iloc[0] if s.notna().any() else None,
        })
    return pd.DataFrame(rows)
prof = profile(raw)
prof.assign(missing_rate=prof["missing_rate"].map("{:.3%}".format))

In [ ]:
# ── 代码格 8 · 需求卡 1.4 对账：两列全空，imsi 缺 15 个 ──
p = prof.set_index("field")
assert p.loc["tmp0", "n_missing"] == 532_828 and p.loc["tmp1", "n_missing"] == 532_828
assert p.loc["imsi", "n_missing"] == 15
print("✅ 对账通过：tmp0、tmp1 全部为空；imsi 缺 15 个")

In [ ]:
# ── 代码格 9 · 需求卡 1.5：时间戳转北京时间 ──
# 这一格在做什么：把 13 位的时间戳转成能看懂的北京时间。
#   unit="ms"      原始值是「毫秒」数（1538498482636）。写成 unit="s" 会得到公元 50000 多年。
#   + 8 小时       时间戳是 UTC（世界标准时间），中国在东八区，要加 8 小时才是本地时间。
# 讲台提示：忘了 +8 小时，图照样画得出来，只是凌晨低谷会跑到下午。
# 让学生自己用“人的作息”去验证——这是本课程教的第一个“用常识检验数据”的例子。
raw["ts"] = pd.to_datetime(raw["timestamp"], unit="ms") + pd.Timedelta(hours=8)
raw[["timestamp", "ts"]].head(3)               # 左边原始值、右边转换结果，放一起看才看得出对不对

In [ ]:
# ── 代码格 10 · 需求卡 1.5：数据从哪天到哪天 ──
raw["ts"].agg(["min", "max"])

In [ ]:
# ── 代码格 11 · 需求卡 1.5 对账：时间范围 ──
assert str(raw["ts"].min())[:16] == "2018-10-01 12:52" and str(raw["ts"].max())[:16] == "2018-10-03 23:57"
print("✅ 对账通过：2018-10-01 12:52 至 2018-10-03 23:57")

In [ ]:
# ── 代码格 12 · 需求卡 1.6：每天有多少数据（中间表写回库里）──
from signal_mysql import database as mysql
# 这一格在做什么：先算出「每天多少条、多少人」这张中间表，再写回库里。
# 讲台提示：本课程的固定套路是「先算中间表 → 写回库 → 画图时从库里取」，而不是边画边算。
# 三个好处：① 图和表用的一定是同一份数，不会对不上；② 别人能直接查这张表复核；
# ③ 改样式重画图时不用重算。企业里报表和看板都是这么分层的。
daily = (raw.assign(day=raw["ts"].dt.strftime("%m-%d"))
            .groupby("day")
            .agg(n_records=("timestamp", "size"), n_users=("imsi", "nunique"))
            .reset_index())
daily["records_share"] = daily["n_records"] / daily["n_records"].sum()
# 核对数：中间表的合计必须等于明细行数。分组统计最常见的错是“悄悄漏了一部分”
# （比如日期是空值的行会被 groupby 默默丢掉），加一行断言就能当场发现。
assert daily["n_records"].sum() == len(raw)
# if_exists="replace"：同一天重跑会整张覆盖，不会越跑越多。这就是「幂等」，D5 会专门讲。
mysql.write_frame(daily, "ads_daily_volume", con, if_exists="replace", index=False)
daily

In [ ]:
# ── 代码格 13 · 需求卡 1.6：画 D1_01「每日记录数」──
from signal_mysql import database as mysql
# 这一格在做什么：画 D1_01「每日记录数」。
# 一张合规的图由五部分组成（见 docs/图表规范.md 第 1 节）：
#   标题   —— 写结论、带数字，数字由代码算出来（下面的 f-string）
#   副标题 —— 写口径：统计对象、时间范围、有没有去重
#   图形   —— 只画回答标题所需要的内容
#   标注   —— 数值直接写在图上，尽量不用图例
#   来源行 —— 左下角浅灰小字
d = mysql.read_frame("SELECT * FROM ads_daily_volume", con)   # 从库里的中间表取，不用上面的 daily 变量
top = d.loc[d["n_records"].idxmax()]                          # 数据最多的那一天，待会儿要强调它
fig, ax = plt.subplots(figsize=(6.4, 4.6), dpi=100)
# 配色规则：灰色打底，只给要强调的那一根上主色——“一种颜色强调一件事”
colors = [MAIN if x == top["day"] else GRAY for x in d["day"]]
bars = ax.bar(d["day"], d["n_records"], color=colors, width=0.6)
for b, x in zip(bars, d["day"]):
    if x != top["day"]:
        b.set_hatch("////")                   # 斜线 = 当天数据不完整
        b.set_edgecolor("white")
ax.bar_label(bars, labels=[f"{v:,}" for v in d["n_records"]], padding=4, color=INK)
ax.set_ylim(0, d["n_records"].max() * 1.15)   # 顶上留 15% 空间，柱顶的数字才不会贴边
# 柱状图的纵轴必须从 0 开始（柱子的长度就是数值，截断会误导）。
# 每根柱都标了数，刻度就是多余的，去掉更干净。
ax.set_yticks([])
ax.grid(False)
# 标题写结论、不写主题，而且数字必须由代码算（f-string）：数据一变，标题跟着变。
# 手敲数字会被下一格的「读回断言」当场抓住——这是课程故意设计的约束。
title = f"{top['records_share']:.1%} 的记录集中在 10 月 {int(top['day'][-2:])} 日"
fig.add_artist(plt.Rectangle((0.02, 0.96 + 0.03 / fig.get_figheight()), 0.38 / fig.get_figwidth(), 0.045 / fig.get_figheight(),
                             transform=fig.transFigure, color=MAIN, lw=0))    # 强调色短条（手册 7.2 部件 1）
fig.text(0.02, 0.96, title, fontsize=16, fontweight="bold", color=INK, va="top")
fig.text(0.02, 0.875, "每日信令记录数（北京时间）；斜线 = 数据不完整；未去重", fontsize=10.5, color=MUTED, va="top")
fig.text(0.02, 0.018, SOURCE, fontsize=8.5, color=FAINT)
fig.tight_layout(rect=(0.01, 0.06, 0.99, 0.83))   # 上面给标题区留位置，下面给来源行留位置
fig.savefig(IMG_DIR / "D1_01_每日记录数.png")